# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** In Silico Computational Biology & Protein Topology

---
*Nota Institucional: Este notebook ilustra la aplicación directa de vectores orientados a objetos en data science sobre la estructura tridimensional de una macromolécula. Transformando  coordenadas atómicas estáticas en una matriz de distancias relativas, se aísla el comportamiento estacional de la densidad proteica —una abstracción matemática equivalente al mapeo de sitios activos biológicos.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("mako")

### 1. Ingestión Directa de Coordenadas Atómicas (.pdb)
Enfocaremos nuestro análisis matricial *in silico* aislando exclusivamente los átomos Carbono-Alfa (CA), los cuales representan geométricamente los vértices funcionales del esqueleto de aminoácidos en la proteína (1UBQ: Ubiquitina).

In [ ]:
def parse_pdb_coordinates(filepath):
    """
    Extracts 3D coordinates specifically for Alpha-Carbons (CA)
    from a raw Protein Data Bank file, disregarding non-structural metadata.
    """
    ca_atoms = []
    with open(filepath, 'r') as file:
        for line in file:
            # Coordinate records in PDB format always start with ATOM
            if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                residue_num = int(line[22:26].strip())
                residue_name = line[17:20].strip()
                
                # Orthogonal coordinates for X, Y, Z in Angstroms
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                
                ca_atoms.append({
                    'Residue_Num': residue_num,
                    'Residue_Name': residue_name,
                    'X': x, 'Y': y, 'Z': z
                })
                
    return pd.DataFrame(ca_atoms)

df_ca = parse_pdb_coordinates('../data/1ubq.pdb')
print(f"[*] Extraídos {len(df_ca)} nodos estructurales (Átomos Carbono-Alfa).")
df_ca.head()

### 2. Transformación Espacial: Matriz de Distancia Euclidiana
Al permutar las posiciones cartesianas contra sí mismas produciendo una matriz simétrica $N \times N$, convertimos la proteína tridimensional en un grafo 2D de cercanía atómica, identificando zonas funcionalmente densas.

In [ ]:
coords = df_ca[['X', 'Y', 'Z']].values

# Vectorized euclidean distance matrix calculation
dist_mat = distance_matrix(coords, coords)

# Create Heatmap for visual clustering
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    dist_mat, 
    cmap="viridis_r", # Reversed so darker equals closer (high density)
    square=True, 
    cbar_kws={'label': 'Distancia Geométrica (Ángstroms)'},
    ax=ax
)

ax.set_title("Topología Proteica (1UBQ): Densidad de Cadenas Laterales", fontsize=14, pad=15)
ax.set_xlabel("Índice del Residuo Aminoácido")
ax.set_ylabel("Índice del Residuo Aminoácido")

plt.show()

### Conclusión Topológica
Lo que en el plano biológico representaba el complejo plegamiento globular de la ubicuitina, en la matriz de calor se traduce en anomalías de alta intensidad fuera de su diagonal principal. Estas bandas (distancias cartesianas cercanas a $0Å$) mapean algorítmicamente la compresión de regiones distantes de la cadena primaria que se enlazan espaciadas formando posibles **puentes de hidrógeno** y el centro hidrofóbico funcional.